# Notebook 03 — Decision Tree
## Learning Rules as Yes/No Questions

**Dataset**: Titanic passengers
**Question we answer**: *Did this passenger survive?*
**Learning type**: Supervised Classification — rules expressed as a flowchart

---

## Section 1 — What Is a Decision Tree?

### In plain English

Think of how a doctor diagnoses a patient using a flowchart:

```
Is the patient female?
├── Yes → What cabin class?
│         ├── 1st or 2nd → Likely survived
│         └── 3rd        → Check family size...
└── No  → What cabin class?
          ├── 1st         → Check age...
          └── 2nd or 3rd  → Likely did not survive
```

**A decision tree learns exactly this kind of flowchart automatically from the data.**

At each step it asks the single most useful yes/no question to separate survivors from non-survivors.
It keeps asking questions until each group is "pure" — mostly all survived, or mostly all did not.

### The tree IS the rules

In logistic regression, rules are hidden inside weights in a formula.
In a decision tree, **you can print the tree and read every decision it makes**.
This makes decision trees the most interpretable of all four methods.

## Section 2 — How Does It Learn?

### Finding the best question at each node

The tree tries every possible split on every feature and picks the one that creates the **purest** groups.

Purity is measured by **Gini Impurity**:
```
Gini = 1 − (fraction_survived² + fraction_not_survived²)
```
- Node with 100% survivors: Gini = 1 − (1² + 0²) = **0.0** ← perfectly pure ✓
- Node with 50/50 split:    Gini = 1 − (0.5² + 0.5²) = **0.5** ← maximum impurity ✗

**Learning process:**
1. Start with all passengers at the root
2. Try every feature, every threshold: "Is age < 15?", "Is sex female?", "Is pclass = 1?"
3. Pick the split with the lowest weighted Gini across both child nodes
4. Recurse on each branch until pure or maximum depth reached

### Key insight: no distances, no gradients, no scaling

The tree only asks "is this value above or below threshold Y?"
Whether a feature ranges 0–1 or 0–1000 **does not affect** the split decision.

## Section 3 — What Does the Data Need?

| Requirement | Linear Regression | Logistic Regression | **Decision Tree** |
|---|---|---|---|
| Feature scaling | **Required** | **Required** | Not needed |
| Numeric encoding | Required | Required | Required |
| Missing values | Not allowed | Not allowed | Not allowed (sklearn) |
| Distribution shape | Matters | Matters | **Irrelevant** |

### Per-feature prep for trees

| Feature | Transform | Why |
|---|---|---|
| `pclass` | None | Raw ordinal 1/2/3 works fine for threshold splits |
| `sex_enc` | Encode 0/1 | Trees need numbers, not strings |
| `age` | Fill median | No NaN allowed |
| `sibsp` | None | Raw counts work fine |
| `parch` | None | Raw counts work fine |
| `log_fare` | Optional | Log reduces outlier effect on splits |
| `embarked_enc` | Encode 0/1/2 | Trees need numbers |
| `family_size` | None | Raw counts work fine |

> Applying StandardScaler to a decision tree produces identical results.
> Scaling is invariant for threshold-based splits — it is just wasted computation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

df_raw = sns.load_dataset('titanic')

# Simpler prep — no scaling step needed
df = df_raw.copy()
df['age']          = df['age'].fillna(df['age'].median())
df['fare']         = df['fare'].fillna(df['fare'].median())
df['embarked']     = df['embarked'].fillna(df['embarked'].mode()[0])
df['sex_enc']      = (df['sex'] == 'female').astype(int)
df['embarked_enc'] = df['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df['family_size']  = df['sibsp'] + df['parch'] + 1
df['log_fare']     = np.log1p(df['fare'])
df = df.dropna(subset=['survived'])

FEATURES = ['pclass','sex_enc','age','sibsp','parch','log_fare','embarked_enc','family_size']
TARGET   = 'survived'

X = df[FEATURES].values
y = df[TARGET].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train)} passengers  |  Test: {len(X_test)} passengers")
print("No StandardScaler applied — decision trees do not need it.")

## Section 4 — Train the Model and Read the Rules

A **shallow tree** (max_depth=3) is easy to read — you can trace every path.
A **deep tree** memorises the training data but generalises poorly (overfitting).

We train both, compare accuracy, then visualise the shallow tree.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree3.fit(X_train, y_train)
acc3 = accuracy_score(y_test, tree3.predict(X_test))

tree8 = DecisionTreeClassifier(max_depth=8, random_state=42)
tree8.fit(X_train, y_train)
acc8 = accuracy_score(y_test, tree8.predict(X_test))

print(f"Depth-3 tree accuracy : {acc3*100:.1f}%  (simple, readable)")
print(f"Depth-8 tree accuracy : {acc8*100:.1f}%  (complex, risk of overfitting)")
print()
print("Rules learned by the depth-3 tree:")
print(export_text(tree3, feature_names=FEATURES))

In [ ]:
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(tree3, feature_names=FEATURES,
          class_names=['Not survived','Survived'],
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree (max_depth=3) — read the splitting rule in each box',
             fontsize=12, pad=15)
plt.tight_layout()
plt.show()
print('Blue boxes  → model predicts: Survived')
print('Orange boxes → model predicts: Not survived')
print('Darker colour = purer node = more confident prediction')

In [ ]:
importance_df = pd.DataFrame({
    'Feature'    : FEATURES,
    'Importance' : tree3.feature_importances_.round(4),
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
ax.set_title('Feature Importances (depth-3 Decision Tree)\nHigher = used more often for splitting', fontsize=12)
ax.set_xlabel('Importance (weighted Gini reduction)', fontsize=11)
plt.tight_layout()
plt.show()